<a href="https://colab.research.google.com/github/Palash0306/credit-risk-scoring-engine/blob/develop/01_ingest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Colab → click the key icon (Secrets) on left sidebar
# Add these three secrets:
# AWS_ACCESS_KEY_ID     → your access key
# AWS_SECRET_ACCESS_KEY → your secret key
# S3_BUCKET             → credit-risk-engine-prod-2026

from google.colab import userdata
import os

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
BUCKET = userdata.get('S3_BUCKET')

print("Credentials loaded from Colab Secrets ✅")
print(f"Bucket: {BUCKET}")

Credentials loaded from Colab Secrets ✅
Bucket: credit-risk-engine-prod-2026


In [2]:
!pip install pyspark>=4.0.0 delta-spark>=4.0.0 boto3 findspark -q
print("Dependencies installed ✅")

import findspark
findspark.init()

print("Environment successfully locked down to PySpark 4.0.3! 🛡️")

Dependencies installed ✅
Environment successfully locked down to PySpark 4.0.3! 🛡️


In [3]:
import pyspark
print(pyspark.__version__)

4.0.3


Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "/content/drive/MyDrive/credit-risk-engine"
os.makedirs(f"{DRIVE}/delta", exist_ok=True)

print(f"Drive mounted at {DRIVE} ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive/MyDrive/credit-risk-engine ✅


Verify S3 files

In [5]:
import boto3

s3 = boto3.client('s3')
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="raw/")

if 'Contents' in response:
    for obj in response['Contents']:
        print(f"{obj['Key']}  —  {obj['Size']/1e6:.1f} MB")
else:
    print("No files found")
print("S3 verified ✅")

raw/  —  0.0 MB
raw/home_credit.csv  —  166.1 MB
raw/lending_club.csv  —  1675.1 MB
S3 verified ✅


Start Spark with Delta Lake

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CreditRiskIngestion") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.13:4.0.0,"
            "org.apache.hadoop:hadoop-aws:3.4.0") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.access.key",
            os.environ['AWS_ACCESS_KEY_ID']) \
    .config("spark.hadoop.fs.s3a.secret.key",
            os.environ['AWS_SECRET_ACCESS_KEY']) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

print(f"Spark {spark.version} ready ✅")
print("Delta Lake ready ✅")

Spark 4.0.3 ready ✅
Delta Lake ready ✅


Read Lending Club from S3 (sample first)


In [7]:
import pandas as pd
import io

print("Reading Lending Club from S3...")
obj = s3.get_object(Bucket=BUCKET, Key="raw/lending_club.csv")

df_lc = pd.read_csv(
    io.BytesIO(obj['Body'].read()),
    nrows=100000,
    low_memory=False
)

print(f"Rows    : {len(df_lc):,}")
print(f"Columns : {len(df_lc.columns)}")

Reading Lending Club from S3...
Rows    : 100,000
Columns : 151


In [8]:
df_lending = spark.createDataFrame(df_lc)
print("Spark DataFrame ready ✅")

Spark DataFrame ready ✅


Write Lending Club to Delta Lake


In [9]:
df_lending.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{DRIVE}/delta/raw_loans")

print("raw_loans written to Delta Lake ✅")

raw_loans written to Delta Lake ✅


Read Home Credit from S3

In [10]:
print("Reading Home Credit from S3...")
obj_hc = s3.get_object(Bucket=BUCKET, Key="raw/home_credit.csv")

df_hc = pd.read_csv(
    io.BytesIO(obj_hc['Body'].read()),
    nrows=50000,
    low_memory=False
)

print(f"Rows    : {len(df_hc):,}")
print(f"Columns : {len(df_hc.columns)}")
df_home = spark.createDataFrame(df_hc)

# df_home.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save(f"{DRIVE}/delta/home_credit_raw")

# print("home_credit_raw written to Delta Lake ✅")

Reading Home Credit from S3...
Rows    : 50,000
Columns : 122


Verify both Delta tables

In [11]:
df_check_lc = spark.read.format("delta").load(f"{DRIVE}/delta/raw_loans")
df_check_hc = spark.read.format("delta").load(f"{DRIVE}/delta/home_credit_raw")

print(f"raw_loans      : {df_check_lc.count():,} rows × {len(df_check_lc.columns)} cols")
print(f"home_credit_raw: {df_check_hc.count():,} rows × {len(df_check_hc.columns)} cols")
print("\nPhase 1 ingestion complete ✅")

raw_loans      : 100,000 rows × 151 cols
home_credit_raw: 50,000 rows × 122 cols

Phase 1 ingestion complete ✅
